In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scipy.stats as stats

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

from sklearn.preprocessing import PowerTransformer

In [ ]:
df = pd.read_csv("./concrete_data.csv")
df.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


In [5]:
X = df.drop("Strength", axis=1)
y = df["Strength"]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

### 1. Without applying any transformation

In [8]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

r2_score(y_test, y_pred)

0.627553179231485

In [ ]:
# Cross Val Score
lr = LinearRegression()
np.mean(cross_val_score(lr, X, y, scoring="r2"))

np.float64(0.46099404916628606)

### 2. Applying Box-Cox Transform

In [10]:
# We are estimating lamda here for each column
pt = PowerTransformer(method="box-cox")

X_train_transformed = pt.fit_transform(X_train + 0.00001)
X_test_transformed = pt.fit_transform(X_test + 0.00001)

pd.DataFrame({"cols": X_train.columns, "box_cox_lambdas": pt.lambdas_})

,cols,box_cox_lambdas
0,Cement,0.215602
1,Blast Furnace Slag,0.028899
2,Fly Ash,-0.007561
3,Water,0.959062
4,Superplasticizer,0.119398
5,Coarse Aggregate,1.192491
6,Fine Aggregate,1.973781
7,Age,-0.014692


In [11]:
## Applying LR on transformed data

lr = LinearRegression()
lr.fit(X_train_transformed, y_train)

y_pred2 = lr.predict(X_test_transformed)
r2_score(y_test, y_pred2)

0.8061415974066064

In [13]:
# Cross Val Score

pt = PowerTransformer(method="box-cox")
X_transformed = pt.fit_transform(X + 0.00001)

lr = LinearRegression()
np.mean(cross_val_score(lr, X_transformed, y, scoring="r2"))

np.float64(0.6668489648058483)